# 02 - Train Emotion-Specific QLoRA Adapters

Trains 4 QLoRA adapters on Mistral-7B-Instruct, one per emotion cluster.

**Runtime:** set to GPU (A100) + High RAM in Colab via Runtime > Change runtime type.

## 1. Setup

In [ ]:
# clone repo and install deps
import os

if not os.path.exists("/content/NLP-Project/src"):
    !git clone https://github.com/marcnasrisme/NLP-Project.git /content/NLP-Project
else:
    !cd /content/NLP-Project && git pull

%cd /content/NLP-Project
!pip install -q -r requirements.txt

In [ ]:
# mount google drive for persistent checkpoints
from google.colab import drive
drive.mount('/content/drive')

ADAPTER_OUTPUT = "/content/drive/MyDrive/NLP_Project_adapters"
os.makedirs(ADAPTER_OUTPUT, exist_ok=True)
print(f"adapters will be saved to: {ADAPTER_OUTPUT}")

In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Load and prepare data

In [ ]:
from src.data.load import load_splits
from src.data.cluster import build_emotion_clusters, save_clusters
from src.data.format import format_for_sft

splits = load_splits()
print(f"loaded: {len(splits['train'])} train, {len(splits['validation'])} val, {len(splits['test'])} test conversations")

In [ ]:
# cluster emotions and save config
cluster_result = build_emotion_clusters()
save_clusters(cluster_result, "configs/emotion_clusters.yaml")

emotion_to_cluster = cluster_result["emotion_to_cluster"]
cluster_names = cluster_result["cluster_names"]

for cid, name in cluster_names.items():
    emotions = cluster_result["clusters"][name]["emotions"]
    print(f"Cluster {cid} ({name}): {len(emotions)} emotions")

In [ ]:
# format all training examples
train_examples = format_for_sft(splits["train"], emotion_to_cluster)
print(f"total training examples: {len(train_examples)}")

# split by cluster
cluster_data = {}
for cid, name in cluster_names.items():
    cluster_data[cid] = [ex for ex in train_examples if ex["cluster_id"] == cid]
    print(f"  cluster {cid} ({name}): {len(cluster_data[cid])} examples")

## 3. Load base model

This loads Mistral-7B-Instruct in 4-bit. Takes ~2 minutes on first run (downloads ~14.5GB, quantized to ~5GB in memory).

In [ ]:
from src.models.adapter import load_config, load_base_model

config = load_config("configs/adapter_training.yaml")
base_model, tokenizer = load_base_model(config)
print(f"model loaded: {config['model']['name']}")
print(f"tokenizer vocab size: {len(tokenizer)}")

## 4. Train adapters

Trains one adapter per cluster, saved to Google Drive.

If Colab dies mid-training, just re-run all cells from the top. Training resumes from the last checkpoint on Drive.

**To train specific clusters**, change `clusters_to_train` below (e.g. `[1, 2, 3]` to skip cluster 0).

In [ ]:
# change this to train specific clusters, e.g. [1, 2, 3] to skip already-trained ones
# set to None to train all 4
clusters_to_train = None

In [ ]:
from src.models.adapter import attach_adapter, train_adapter

target_clusters = clusters_to_train or list(cluster_names.keys())

for cid in target_clusters:
    name = cluster_names[cid]
    output_dir = f"{ADAPTER_OUTPUT}/cluster_{cid}_{name}"
    examples = cluster_data[cid]

    print(f"\n{'='*60}")
    print(f"Training adapter for cluster {cid}: {name}")
    print(f"  examples: {len(examples)}")
    print(f"  output: {output_dir}")
    print(f"{'='*60}\n")

    model = attach_adapter(base_model, config)

    train_adapter(
        model=model,
        tokenizer=tokenizer,
        train_examples=examples,
        output_dir=output_dir,
        config=config,
    )

    base_model = model.unload()
    print(f"\nAdapter {cid} ({name}) saved to {output_dir}")

## 5. Verify saved adapters

In [ ]:
for cid, name in cluster_names.items():
    adapter_dir = f"{ADAPTER_OUTPUT}/cluster_{cid}_{name}"
    if os.path.exists(adapter_dir):
        files = os.listdir(adapter_dir)
        has_weights = "adapter_model.safetensors" in files or "adapter_model.bin" in files
        has_config = "adapter_config.json" in files
        status = "ready" if (has_weights and has_config) else "incomplete"
        print(f"Cluster {cid} ({name}): {status}")
    else:
        print(f"Cluster {cid} ({name}): not trained yet")

## 6. Quick test: generate with one adapter

In [ ]:
from src.models.adapter import load_trained_adapter

test_cid = target_clusters[0]
test_name = cluster_names[test_cid]
adapter_path = f"{ADAPTER_OUTPUT}/cluster_{test_cid}_{test_name}"

model_with_adapter = load_trained_adapter(base_model, adapter_path)
model_with_adapter.eval()

prompt = "[INST] Situation: I just lost my job.\nEmotion: devastated\n\nSpeaker: I got laid off today. I don't know what to do. [/INST]"
inputs = tokenizer(prompt, return_tensors="pt").to(model_with_adapter.device)

with torch.no_grad():
    output = model_with_adapter.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        do_sample=True,
    )

response = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(f"Adapter: {test_name}")
print(f"Response: {response}")